Deep Q-Network Learning

In [2]:
import torch
import numpy as np
import time, random
import torch.nn as nn
import gymnasium as gym
import torch.nn.functional as F
import matplotlib.pyplot as plt
from dataclasses import dataclass

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
@dataclass
class Config:
    env_id: str = "CartPole-v1"
    total_steps: int = 60_000
    buffer_size: int = 50_000
    batch_size: int = 64
    gamma: float = 0.99
    lr: float = 1e-3
    hidden: int = 128
    learning_starts: int = 1_000 # warmup for buffer init
    train_every: int = 1 # env steps per gradient step
    target_sync_every: int = 500 # gradient steps per hard target copy
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_decay_steps: int = 10_000    # linear decay
    eval_window: int = 100
    solve_threshold: float = 475.0   # CartPole-v1: mean return >= 475 over 100 episodes

CFG = Config()
Q_CEILING = (1 - CFG.gamma ** 500)/(1 - CFG.gamma)

In [5]:
Q_CEILING

99.34295169575846

Replay Buffer

In [6]:
class ReplayBuffer:
    """FIFO ring buffer over np arrays. Stores only terminated episodes"""
    def __init__(self, capacity: int, obs_dim: int):
        self.capacity = capacity
        self.states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.size = 0

    def add(self, state, action, reward, next_state, terminated):
        i = self.pos
        self.states[i] = state
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_states[i] = next_state
        self.terminated[i] = terminated
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self.size, size=batch_size)
        pick = lambda arr, dtype: torch.as_tensor(arr[idx], dtype=dtype, device=device)
        return (
            pick(self.states, torch.float32),
            pick(self.actions, torch.int64),
            pick(self.rewards, torch.float32),
            pick(self.next_states, torch.float32),
            pick(self.terminated, torch.float32),
        )

    def __len__(self):
        return self.size

Q-Network and Epsilon Schedule

In [7]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, n_hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_actions)
        )

    def forward(self, x):
        return self.net(x)

Epsilon Decay

In [8]:
def epsilon_at(step: int, cfg: Config) -> float:
    frac = min(step / cfg.eps_decay_steps, 1.0)
    return cfg.eps_start + frac * (cfg.eps_end - cfg.eps_start)